# Smart Chatbot with Memory and Feedback

This notebook extends a basic BlenderBot chatbot with:
- Conversation context during the current chat
- Simple personal memory saved in a local JSON file
- Helpful / not-helpful feedback saved locally
- A Gradio chat interface compatible with newer Gradio versions

**Important:** This is a learning demo. Do not save sensitive information such as passwords or financial details. The memory file is local and this simple version is intended for one user at a time.

## 1. Install the libraries

In [ ]:
!pip install -q -U transformers torch gradio

## 2. Load the chatbot model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

MODEL_NAME = "facebook/blenderbot-400M-distill"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded on:", device)

## 3. Set up local memory and feedback files

In [ ]:
import json
from pathlib import Path
from datetime import datetime

MEMORY_FILE = Path("user_memory.json")
FEEDBACK_FILE = Path("chatbot_feedback.json")

def load_json(path, default):
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            return default
    return default

def save_json(path, data):
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

user_memory = load_json(MEMORY_FILE, {})
feedback_log = load_json(FEEDBACK_FILE, [])

print("Local memory and feedback storage are ready.")

## 4. Add simple memory commands

Try these messages in the chat:
- `remember my name is Arun`
- `remember I am learning Python`
- `what do you remember about me?`
- `forget my name`

The memory system recognizes these simple patterns; it does not understand every possible wording.

In [ ]:
def handle_memory_command(message):
    text = message.strip()
    lower = text.lower()

    prefix = "remember my name is "
    if lower.startswith(prefix):
        name = text[len(prefix):].strip()
        if name:
            user_memory["name"] = name
            save_json(MEMORY_FILE, user_memory)
            return f"I'll remember your name is {name}."

    prefix = "remember i am "
    if lower.startswith(prefix):
        fact = text[len(prefix):].strip()
        if fact:
            user_memory["about_me"] = fact
            save_json(MEMORY_FILE, user_memory)
            return f"I'll remember: {fact}."

    if lower in {"what do you remember about me", "what do you remember about me?"}:
        if not user_memory:
            return "I haven't saved any personal facts yet."
        facts = "; ".join(
            f"{key.replace('_', ' ')}: {value}"
            for key, value in user_memory.items()
        )
        return "Here is what I remember: " + facts

    if lower == "forget my name":
        user_memory.pop("name", None)
        save_json(MEMORY_FILE, user_memory)
        return "I've removed your saved name, if one was stored."

    return None

## 5. Generate a chatbot response

In [ ]:
def generate_reply(user_message, history):
    # Handle supported memory commands first.
    memory_reply = handle_memory_command(user_message)
    if memory_reply is not None:
        return memory_reply

    # Add saved facts as lightweight context.
    context = ""
    if user_memory.get("name"):
        context += f"The user's name is {user_memory['name']}. "
    if user_memory.get("about_me"):
        context += f"The user shared this information: {user_memory['about_me']}. "

    # Newer Gradio Chatbot components use message dictionaries:
    # {"role": "user"/"assistant", "content": "..."}
    recent_history = history[-8:] if history else []
    conversation_text = context

    for item in recent_history:
        if not isinstance(item, dict):
            continue
        role = item.get("role", "")
        content = item.get("content", "")
        if not isinstance(content, str):
            continue
        if role == "user":
            conversation_text += f"User: {content}\n"
        elif role == "assistant":
            conversation_text += f"Bot: {content}\n"

    conversation_text += f"User: {user_message}\nBot:"

    inputs = tokenizer(
        conversation_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            top_p=0.9,
            temperature=0.7
        )

    reply = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    if not reply:
        reply = "Sorry, I couldn't form a reply. Could you ask another way?"
    return reply

## 6. Add the Gradio chat and feedback interface

In [ ]:
import gradio as gr

def save_feedback(history, rating):
    last_user = ""
    last_bot = ""

    # Find the latest user message and assistant reply.
    for item in reversed(history or []):
        if not isinstance(item, dict):
            continue
        if item.get("role") == "assistant" and not last_bot:
            last_bot = item.get("content", "")
        elif item.get("role") == "user" and not last_user:
            last_user = item.get("content", "")
        if last_user and last_bot:
            break

    if not history:
        return "Send a message before rating a response."

    record = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "rating": rating,
        "user_message": last_user,
        "bot_reply": last_bot
    }
    feedback_log.append(record)
    save_json(FEEDBACK_FILE, feedback_log)
    return f"Thanks! Your feedback was saved as: {rating}."

def user_submit(user_message, history):
    history = history or []
    if not user_message or not user_message.strip():
        return "", history

    message_text = user_message.strip()
    reply = generate_reply(message_text, history)

    # New Gradio message format: one dictionary per message.
    updated_history = history + [
        {"role": "user", "content": message_text},
        {"role": "assistant", "content": reply}
    ]
    return "", updated_history

with gr.Blocks() as demo:
    gr.Markdown("# Smart Chatbot with Memory and Feedback")
    gr.Markdown(
        "Try: `remember my name is Arun`, `remember I am learning Python`, "
        "`what do you remember about me?`, or `forget my name`."
    )

    chatbot = gr.Chatbot(label="Conversation")
    message = gr.Textbox(
        label="Your message",
        placeholder="Type a message and press Enter..."
    )
    feedback_status = gr.Markdown()

    with gr.Row():
        send_btn = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("Clear conversation")
        good_btn = gr.Button("👍 Helpful")
        bad_btn = gr.Button("👎 Not helpful")

    send_btn.click(
        user_submit,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    )
    message.submit(
        user_submit,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    )
    clear_btn.click(lambda: [], inputs=None, outputs=chatbot)

    good_btn.click(
        lambda hist: save_feedback(hist, "helpful"),
        inputs=[chatbot],
        outputs=[feedback_status]
    )
    bad_btn.click(
        lambda hist: save_feedback(hist, "not helpful"),
        inputs=[chatbot],
        outputs=[feedback_status]
    )

demo.launch(share=False)

## 7. Test the notebook

1. Run all cells from top to bottom.
2. In the chat, send `remember my name is Arun`.
3. Ask `what do you remember about me?`.
4. Send `remember I am learning Python`.
5. Ask a follow-up question.
6. Click **Helpful** or **Not helpful** to save feedback.
7. Check the files `user_memory.json` and `chatbot_feedback.json`.

## Notes and limitations

- BlenderBot may not always follow the supplied context accurately.
- The memory commands only recognize the example patterns above.
- The local memory file is shared by this notebook session and is not separated by user accounts.
- Feedback is saved for later review; it does not automatically retrain the model.
- In hosted notebooks, local files may disappear when the runtime is reset unless you save them elsewhere.